# P4 · Диагностика `--m_message_feature` на genre: что ломает скор-колонка в forward

**Наблюдение.** На trade фича накидывает, на genre — просаживает: countloader 0.492 → +mmsg 0.486 (⚠️ и это ещё неравноконфигно: mmsg-раны шли при κ=40/25 и реапнуты на e5–7, контроль κ=25/10эп; но вопрос «не должно быть хуже» валиден).

**Гипотеза (из ревью имплементации).** Скор-колонка — **сырой** decayed `M[u,i]`: (а) **масштаб** O(10–100) у тяжёлых юзеров рядом с O(1) колонками (`raw_msg`, cos-энкодинги в [−1,1]) — одна горячая колонка из 3922 во входе GRU и из 786 в edge_attr; (б) **нестационарность**: M сбрасывается каждую train-эпоху и растёт монотонно train→val→test без сброса — eval видит магнитуды, которых supervised-шаги в начале эпохи не видели. На trade сырая магнитуда — почти сама метка (label = нормированные объёмы года), поэтому там колонка помогает; на genre сигнал в **ранжировании**, а магнитуда — конфаундер активности юзера.

**План.** (1) Распределение скора по ходу стрима (нестационарность + масштаб vs другие фичи); (2) forward-хуки на паре чекпойнтов κ=25: ON (`_mmsg`, e5) vs OFF (e10) — распределения скрытых состояний по слоям (message-вектор по группам колонок, GRU-апдейт, память, edge_attr, вклад скор-колонки в `lin_edge`, логиты); (3) вердикт + фикс (сквош-трансформа), сохраняющий выигрыш trade.

In [1]:
# Ячейка 1 — масштаб/нестационарность скор-колонки по ходу стрима (MSampler κ=25, как в чекпойнтах)
import numpy as np, polars as pl, torch, math, time
import plotly.express as px, plotly.graph_objects as go
from tgb.nodeproppred.dataset_pyg import PyGNodePropPredDataset
from torch_geometric.loader import TemporalDataLoader
import sys; sys.path.insert(0, "..") if ".." not in sys.path else None
from models.msampler import MSampler

ds = PyGNodePropPredDataset(name="tgbn-genre", root="datasets")
data = ds.get_TemporalData()
tr_d, va_d, te_d = data.train_val_test_split(val_ratio=0.15, test_ratio=0.15)
N, C = data.num_nodes, ds.num_classes
kappa = 25.0
smp = MSampler(N, C, 10, math.log(2)/(kappa*86400.0), torch.sort(tr_d.msg[:, 0]).values, device="cpu")

rows, day_i = [], 0
t0 = time.time()
def stream(split, dd):
    global day_i
    label_t = ds.get_label_time()
    for b in TemporalDataLoader(dd, batch_size=200):
        if float(b.t[-1]) > label_t:
            lt = ds.get_node_label(b.t[-1])
            if lt is None: break
            label_t = ds.get_label_time(); day_i += 1
            pm = b.t < float(lt[0][0])
            if pm.any():  # memory-канал: скор записываемых рёбер (до insert — как в process_edges)
                sc = smp.score_for(b.src[pm], b.dst[pm], b.t[pm])
                if day_i % 12 == 0:  # каждый 12-й день
                    us = lt[1]
                    out = smp(us)    # GNN-канал: m_e_raw на сэмплах label-юзеров
                    mer = out[4]
                    rows.append((day_i, split, float(sc.median()), float(sc.quantile(0.95)), float(sc.max()),
                                 float(mer.median()) if mer.numel() else 0.0,
                                 float(mer.quantile(0.95)) if mer.numel() else 0.0,
                                 float(mer.max()) if mer.numel() else 0.0))
                smp.insert(b.src[pm], b.dst[pm], b.t[pm], b.msg[pm])
            m2 = ~pm
            if m2.any(): smp.insert(b.src[m2], b.dst[m2], b.t[m2], b.msg[m2])
        else:
            smp.insert(b.src, b.dst, b.t, b.msg)
ds.reset_label_time()
stream("train", tr_d); stream("val", va_d); stream("test", te_d); ds.reset_label_time()
df = pl.DataFrame(rows, schema=["day", "split", "sc_med", "sc_p95", "sc_max", "mer_med", "mer_p95", "mer_max"], orient="row")
print(f"[{time.time()-t0:.0f} c] точек: {len(df)}; сравнение масштабов (последняя треть train / test):")
print(df.filter(pl.col("split") == "train").tail(10).select(["sc_med", "sc_p95", "sc_max", "mer_p95", "mer_max"]).mean())
print(df.filter(pl.col("split") == "test").select(["sc_med", "sc_p95", "sc_max", "mer_p95", "mer_max"]).mean())
print("справка: raw_msg (вес ребра) p95 =", float(tr_d.msg[:, 0].quantile(0.95)), "; cos-энкодинги ∈ [-1, 1]")

[11 c] точек: 127; сравнение масштабов (последняя треть train / test):
shape: (1, 5)
┌──────────┬────────────┬────────────┬────────────┬─────────────┐
│ sc_med   ┆ sc_p95     ┆ sc_max     ┆ mer_p95    ┆ mer_max     │
│ ---      ┆ ---        ┆ ---        ┆ ---        ┆ ---         │
│ f64      ┆ f64        ┆ f64        ┆ f64        ┆ f64         │
╞══════════╪════════════╪════════════╪════════════╪═════════════╡
│ 33.49184 ┆ 404.331232 ┆ 694.215214 ┆ 192.826097 ┆ 1278.537042 │
└──────────┴────────────┴────────────┴────────────┴─────────────┘
shape: (1, 5)
┌───────────┬──────────┬─────────────┬────────────┬────────────┐
│ sc_med    ┆ sc_p95   ┆ sc_max      ┆ mer_p95    ┆ mer_max    │
│ ---       ┆ ---      ┆ ---         ┆ ---        ┆ ---        │
│ f64       ┆ f64      ┆ f64         ┆ f64        ┆ f64        │
╞═══════════╪══════════╪═════════════╪════════════╪════════════╡
│ 40.697666 ┆ 764.9989 ┆ 1266.872345 ┆ 224.573321 ┆ 2450.72563 │
└───────────┴──────────┴─────────────┴───────────

справка: raw_msg (вес ребра) p95 = 1.0 ; cos-энкодинги ∈ [-1, 1]


In [2]:
# Траектория масштаба скор-колонки по дням стрима (оба канала), log-y
pdf = df.to_pandas()
fig = go.Figure()
for col, nm, dash in [("sc_med", "score_for медиана", "solid"), ("sc_p95", "score_for p95", "solid"),
                      ("mer_p95", "m_e_raw p95 (GNN)", "dot"), ("mer_max", "m_e_raw max (GNN)", "dot")]:
    fig.add_scatter(x=pdf["day"], y=pdf[col].clip(lower=1e-2), mode="lines", name=nm, line=dict(dash=dash))
for s, x0 in pdf.groupby("split")["day"].min().items():
    if s != "train": fig.add_vline(x=x0, line_dash="dash", line_color="gray", annotation_text=s)
fig.add_hline(y=1.0, line_color="red", annotation_text="масштаб raw_msg / cos-энкодингов (≈1)", annotation_position="bottom right")
fig.update_layout(title="Скор-колонка vs остальные фичи: на 2–3 порядка горячее и растёт по стриму (log-y)",
                  xaxis_title="label-день (train → val → test)", yaxis_title="величина скора (log)",
                  yaxis_type="log", height=450)
fig.show()

**Шаг 1 — гипотеза (а) и (б) подтверждены на входных данных.** Скор-колонка: медиана ~35, **p95 ≈ 400–770, max до 1267** (memory-канал) и **до 2451** (GNN-канал `m_e_raw`) — при том что соседние фичи ≤ 1 (raw_msg p95 = 1.0, cos-энкодинги ∈ [−1,1]). Это **одна колонка на 2–3 порядка горячее** остальных 3921 во входе GRU и 785 в edge_attr. Нестационарность: p95 растёт train → test на **+89%** (404 → 765), max — ×1.8; в GNN-канале max ×1.9. То есть (i) на инициализации вклад колонки в пре-активации ~в сотни раз больше любой другой фичи → GRU-гейты/`lin_edge` насыщаются или градиент по остальным фичам относительно глохнет; (ii) к test-стриму колонка систематически больше, чем на supervised-шагах — обученный на train-масштабе вес читает смещённый вход. Далее — как это выглядит в скрытых состояниях реальных обученных чекпойнтов.

In [3]:
# Ячейка 4 — веса на скор-колонке в обученном ON-чекпойнте (κ=25, e5) vs OFF (κ=25, e10)
CKPT_ON  = "../saved_models/scheduler_constant_tgnv2_dataset_tgbn-genre_bs_200_lr_0.0001_epochs_50_last_neighbour_10_global_dims_784_sampler_memory_km_10_kappa_25.0_mmsg_seed_1.pt"
CKPT_OFF = "../saved_models/scheduler_constant_tgnv2_dataset_tgbn-genre_bs_200_lr_0.0001_epochs_10_last_neighbour_30_global_dims_784_sampler_memory_km_10_kappa_25.0_seed_1.pt"
on, off = torch.load(CKPT_ON, map_location="cpu"), torch.load(CKPT_OFF, map_location="cpu")
print(f"ON: epoch {on['epoch']}, best_val {on['max_val_score']:.4f} | OFF: epoch {off['epoch']}, best_val {off['max_val_score']:.4f}")
d = 784
# --- GRU (вход = message [z_src(784) z_dst(784) raw_msg(1) (score(1)) src_enc(784) dst_enc(784) t_enc(784)]) ---
for tag, ck, has_sc in [("ON ", on, True), ("OFF", off, False)]:
    W = ck["memory"]["gru.weight_ih"]                      # [3d, msg_dim]
    i_msg = 2 * d                                          # raw_msg колонка
    i_sc = i_msg + 1 if has_sc else None
    cn = W.norm(dim=0)                                     # L2 по колонкам
    blocks = {"z_src": cn[:d].mean(), "raw_msg": cn[i_msg], "cos-блоки (ср.)": torch.cat([cn[:i_msg], cn[i_msg+ (2 if has_sc else 1):]]).mean()}
    line = f"GRU {tag}: |w| raw_msg={cn[i_msg]:.4f}  ср.колонка={blocks['cos-блоки (ср.)']:.4f}"
    if has_sc: line += f"  |w| SCORE={cn[i_sc]:.4f}  → эффективный вклад score (×p95=404) = {float(cn[i_sc])*404:.1f} vs raw_msg (×1) = {float(cn[i_msg]):.3f}"
    print(line)
# --- TransformerConv lin_edge (edge_attr = [rel_t_enc(784) msg(1) (score(1))]) ---
for tag, ck, has_sc in [("ON ", on, True), ("OFF", off, False)]:
    W = ck["gnn"]["conv.lin_edge.weight"]                  # [2*out, edge_dim]
    cn = W.norm(dim=0)
    i_msg = d; i_sc = d + 1 if has_sc else None
    line = f"lin_edge {tag}: |w| raw_msg={cn[i_msg]:.4f}  rel_t ср={cn[:d].mean():.4f}"
    if has_sc: line += f"  |w| SCORE={cn[i_sc]:.4f}  → эффективный вклад score (×p95=193) = {float(cn[i_sc])*193:.1f} vs rel_t-блок целиком ≈ {float(cn[:d].mean())*math.sqrt(d)*0.7:.1f}"
    print(line)

ON: epoch 5, best_val 0.4887 | OFF: epoch 5, best_val 0.4917
GRU ON : |w| raw_msg=1.1004  ср.колонка=1.1073  |w| SCORE=0.9656  → эффективный вклад score (×p95=404) = 390.1 vs raw_msg (×1) = 1.100
GRU OFF: |w| raw_msg=1.0859  ср.колонка=1.1096
lin_edge ON : |w| raw_msg=0.6338  rel_t ср=0.5843  |w| SCORE=0.2291  → эффективный вклад score (×p95=193) = 44.2 vs rel_t-блок целиком ≈ 11.5
lin_edge OFF: |w| raw_msg=0.6446  rel_t ср=0.5841


**Шаг 2 — сеть НЕ нейтрализовала горячую колонку.** В ON-чекпойнте вес скор-колонки в GRU `|w|=0.97` — такой же, как у любой из 3920 остальных (~1.1), т.е. Adam за 5 эпох её не «сжал». Эффективные вклады в пре-активации: **GRU: ~390** (p95-скор × вес) против 1.1 от raw_msg; **lin_edge: ~44** против ~11.5 от всего rel_t-блока (784 колонки). Для max-скоров (1267/2451) — тысячи. Прогноз: пре-активации GRU-гейтов для тяжёлых юзеров глубоко в сатурации (sigmoid/tanh прижаты к ±1) → апдейт памяти таких юзеров определяется почти только скором, остальные 3920 признаков задавлены; а из-за нестационарности (+89% p95 к test) рабочая точка ещё и уезжает между train и eval. Уже видно и в метрике: при равных эпохах (e5 vs e5) ON val 0.4887 < OFF 0.4917. Проверяем сатурацию и скрытые состояния на живом forward.

In [4]:
# Ячейка 6 — живой forward ON vs OFF: прогрев хвостом train + 3 val-дня; сатурация GRU-гейтов,
# декомпозиция пре-активаций (score-колонка vs всё остальное), распределения скрытых состояний.
from models.msgmodule import EncodeIndexModule
from models.mtgn import MTGNMemory, LastAggregator
from models.embmodule import MGraphAttentionEmbedding
from models.decoder import NodePredictor
from sklearn.metrics import ndcg_score
torch.manual_seed(0)

def build(msg_feat, ck):
    mm = EncodeIndexModule(784, msg_feat, 784, 784)
    mem = MTGNMemory(N, msg_feat, 784, 784, 784, message_module=mm, aggregator_module=LastAggregator(mm.out_channels))
    gnn = MGraphAttentionEmbedding(in_channels=784, out_channels=784, msg_dim=msg_feat, time_enc=mem.time_enc, m_edge_dim=0).float()
    head = NodePredictor(in_dim=784, out_dim=C)
    mem.load_state_dict(ck["memory"]); gnn.load_state_dict(ck["gnn"]); head.load_state_dict(ck["node_pred"])
    mem.eval(); gnn.eval(); head.eval(); mem.reset_state()
    return mem, gnn, head

M_ON, G_ON, H_ON = build(2, on); M_OFF, G_OFF, H_OFF = build(1, off)
smp = MSampler(N, C, 10, math.log(2)/(25*86400.0), torch.sort(tr_d.msg[:, 0]).values, device="cpu")

gate_stats = {"ON": [], "OFF": []}  # (frac |preact|>4, доля вклада score в ||preact||) по сэмплам батчей
def gru_probe(tag, mem):
    W_ih, b_ih = mem.gru.weight_ih, mem.gru.bias_ih
    i_sc = 2 * 784 + 1
    def hook(mod, inp, out_):
        x, h = inp[0], inp[1]
        if x.numel() == 0 or torch.rand(1).item() > 0.04: return   # сэмплируем ~4% вызовов
        pre = torch.nn.functional.linear(x, W_ih, b_ih)            # [B, 3d] входная часть пре-активаций
        frac_sat = float((pre.abs() > 4).float().mean())
        if tag == "ON":
            contrib_sc = (x[:, i_sc:i_sc+1] * W_ih[:, i_sc]).norm(dim=1)   # вклад score-колонки
            share = float((contrib_sc / pre.norm(dim=1).clamp(min=1e-9)).median())
        else: share = 0.0
        gate_stats[tag].append((frac_sat, share))
    return mem.gru.register_forward_hook(hook)
h1, h2 = gru_probe("ON", M_ON), gru_probe("OFF", M_OFF)

TAIL = 0.985  # прогреваем моделями только последние 1.5% train (M — всем стримом)
n_tr = tr_d.src.size(0); cut = int(n_tr * TAIL)
t0 = time.time(); ds.reset_label_time(); label_t = ds.get_label_time(); bi = 0
for b in TemporalDataLoader(tr_d, batch_size=200):
    while float(b.t[-1]) > label_t:  # прокрутка label-поинтера (без предсказаний в train)
        if ds.get_node_label(b.t[-1]) is None: break
        label_t = ds.get_label_time()
    if bi * 200 >= cut:
        sc = smp.score_for(b.src, b.dst, b.t).view(-1, 1)
        M_ON.update_state(b.src, b.dst, b.t, torch.cat([b.msg, sc], dim=-1))
        M_OFF.update_state(b.src, b.dst, b.t, b.msg)
    smp.insert(b.src, b.dst, b.t, b.msg); bi += 1
print(f"прогрев: {time.time()-t0:.0f} c ({n_tr-cut} рёбер через обе памяти)")

# --- 3 val-дня: forward обеих моделей на одном сэмпле, сбор распределений ---
res = {k: [] for k in ["mem_ON", "mem_OFF", "z_ON", "z_OFF", "H_ON", "H_OFF", "nd_ON", "nd_OFF"]}
days = 0
for b in TemporalDataLoader(va_d, batch_size=200):
    if float(b.t[-1]) > label_t:
        lt = ds.get_node_label(b.t[-1])
        if lt is None or days >= 3: break
        l0 = float(lt[0][0]); us, ys = lt[1], lt[2].numpy(); label_t = ds.get_label_time(); days += 1
        pm = b.t < l0
        if pm.any():
            sc = smp.score_for(b.src[pm], b.dst[pm], b.t[pm]).view(-1, 1)
            M_ON.update_state(b.src[pm], b.dst[pm], b.t[pm], torch.cat([b.msg[pm], sc], dim=-1))
            M_OFF.update_state(b.src[pm], b.dst[pm], b.t[pm], b.msg[pm])
            smp.insert(b.src[pm], b.dst[pm], b.t[pm], b.msg[pm])
        with torch.no_grad():
            n_id2, ei, e_id, m_e, mer = smp(us)
            for tag, mem, gnn, head, gm in [("ON", M_ON, G_ON, H_ON, torch.cat([data.msg[e_id], mer.view(-1,1)], -1)),
                                            ("OFF", M_OFF, G_OFF, H_OFF, data.msg[e_id])]:
                assoc = torch.empty(N, dtype=torch.long); assoc[n_id2] = torch.arange(n_id2.size(0))
                z, lu = mem(n_id2)
                zg = gnn(z, lu, ei, data.t[e_id].float(), gm.float(), m_e=None)[assoc[us]]
                logits = head(zg); p = torch.softmax(logits, 1)
                ent = float((-p * p.clamp(min=1e-12).log()).sum(1).mean())
                nd = np.mean([ndcg_score(ys[i:i+1], logits[i:i+1].numpy(), k=10) for i in range(len(us)) if ys[i].sum() > 0])
                res[f"mem_{tag}"].append(float(z.norm(dim=1).median())); res[f"z_{tag}"].append(float(zg.norm(dim=1).median()))
                res[f"H_{tag}"].append(ent); res[f"nd_{tag}"].append(float(nd))
        m2 = ~pm
        if m2.any():
            sc = smp.score_for(b.src[m2], b.dst[m2], b.t[m2]).view(-1, 1)
            M_ON.update_state(b.src[m2], b.dst[m2], b.t[m2], torch.cat([b.msg[m2], sc], dim=-1))
            M_OFF.update_state(b.src[m2], b.dst[m2], b.t[m2], b.msg[m2])
            smp.insert(b.src[m2], b.dst[m2], b.t[m2], b.msg[m2])
    else:
        sc = smp.score_for(b.src, b.dst, b.t).view(-1, 1)
        M_ON.update_state(b.src, b.dst, b.t, torch.cat([b.msg, sc], dim=-1))
        M_OFF.update_state(b.src, b.dst, b.t, b.msg)
        smp.insert(b.src, b.dst, b.t, b.msg)
h1.remove(); h2.remove(); ds.reset_label_time()
gs_on = np.array(gate_stats["ON"]); gs_off = np.array(gate_stats["OFF"])
print(f"\nGRU пре-активации (вход. часть, {len(gs_on)} сэмплов): доля |preact|>4 (сатурация): ON={gs_on[:,0].mean():.3f}  OFF={gs_off[:,0].mean():.3f}")
print(f"медианная доля нормы пре-активации от ОДНОЙ score-колонки (ON): {gs_on[:,1].mean():.2%}")
for k in ["mem", "z", "H", "nd"]:
    print(f"{k:>4}: ON={np.mean(res[f'{k}_ON']):.4f}  OFF={np.mean(res[f'{k}_OFF']):.4f}")

прогрев: 26 c (187514 рёбер через обе памяти)



GRU пре-активации (вход. часть, 47 сэмплов): доля |preact|>4 (сатурация): ON=0.095  OFF=0.008
медианная доля нормы пре-активации от ОДНОЙ score-колонки (ON): 24.47%
 mem: ON=17.1099  OFF=17.0631
   z: ON=23.0501  OFF=21.8216
   H: ON=3.8375  OFF=3.8035
  nd: ON=0.5163  OFF=0.5236


## Вердикт: фича реализована корректно, но сырой масштаб колонки ломает GRU-гейты на genre

Цепочка доказательств (всё измерено):
1. **Вход**: скор-колонка на 2–3 порядка горячее остальных фич (p95 ≈ 400–770 vs ≤1) и **нестационарна** (+89% p95 train→test; M сбрасывается каждую train-эпоху, но растёт сквозь val/test без сброса).
2. **Веса**: обученный ON-чекпойнт НЕ занулил колонку (|w|=0.97 ≈ у всех) → эффективный вклад ~390 в пре-активации GRU (vs 1.1 от raw_msg) и ~44 в lin_edge (vs ~11.5 от всего rel_t-блока).
3. **Forward**: сатурация входных пре-активаций GRU **9.5% (ON) vs 0.8% (OFF)** — ×12; медианно **24.5% нормы пре-активации — от одной колонки**; нормы памяти не растут (tanh) — деградация именно через прижатые гейты (апдейт памяти тяжёлых юзеров перестаёт видеть остальные 3920 признаков + затухание градиента при обучении). На 3 val-днях: NDCG ON 0.5163 < OFF 0.5236, при равных эпохах (e5) val ON 0.4887 < OFF 0.4917.

**Почему на trade наоборот помогает**: там label(τ) = нормированные объёмы года τ, т.е. сырая магнитуда M — почти прямой линейный предиктор метки; выгода сигнала перевешивает вред сатурации. На genre сигнал — в **ранжировании** строки, а магнитуда — конфаундер активности юзера: чистый вред.

**Фикс — сквош-трансформа колонки в обеих точках инъекции**: `--m_msg_transform log1p` (монотонная — trade-сигнал сохраняется; масштаб 2451 → 7.8; дрейф +89% → +0.64 аддитивно). Default `raw` — чтобы идущие trade-раны не поменяли семантику при resume; на genre гонять с `log1p`. Слаг получает `_mmsglog` (свой чекпойнт). Следующий шаг, если log1p не добьёт: причинный ECDF-сквош до [0,1] (полная стационарность).

⚠️ Отдельно: текущие «default 0.481 / countloader 0.492 / mmsg 0.486» **неравноконфигны** (mmsg: κ=40 и κ=25, реап на e5–7; контроль κ=25 e10) — честный A/B: κ=40, оба ровно 50 эпох, `--m_msg_transform log1p` vs без флага (контрольная рука уже добавлена в `scripts/run_test_score_as_message.sh`).